# Reading zagg products anonymously: ICESat-2 photons over NEON SERC

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/englacial/zagg/main?urlpath=lab/tree/notebooks/sponsor_read_only.ipynb)

_Runs end-to-end on [Binder](https://mybinder.org/v2/gh/englacial/zagg/main?urlpath=lab/tree/notebooks/sponsor_read_only.ipynb):
**no credentials of any kind** — no Earthdata Login, no AWS account. Everything
is an anonymous read of the public bucket. The Binder image provides
`zagg[analysis]` via the repo's `.binder/` environment._

The companion notebook [`sponsor_end_to_end.ipynb`](sponsor_end_to_end.ipynb)
ran the full zagg pipeline — CMR catalog → shardmap → two aggregations of the
ATL03 photon cloud over the NEON [SERC](https://www.neonscience.org/field-sites/serc)
site — and published the outputs to `s3://sliderule-public-cors/`:

| prefix | contents |
|---|---|
| `zagg-examples/serc/tdigest.zarr` | per-cell **t-digest** sketches of photon height (~10 m HEALPix cells), with the morton **location channel** (issue #87) and the strict-AOI mask (issue #101) |
| `zagg-examples/serc/gain_bias.zarr` | per-chunk **gain/bias 128-bin waveforms** (`waveform_counts` + `offset_h`/`gain_h`) from the *same shardmap* (issue #89) |
| `zagg-index/ATL03/007/` | the granule-keyed **chunk-manifest cache** the first run persisted through the virtual-index seam (issue #160) |

This notebook opens all three anonymously and rebuilds the end-to-end
notebook's read-back analysis.

> **Status note (July 2026):** the prefixes above are populated by the
> end-to-end notebook's fleet run. Until that run has been executed the reads
> below will fail with `NoSuchKey`/404 — this notebook is committed unexecuted
> and will be re-committed rendered once the products are live.

## 1. Open the published stores (anonymous S3)

zagg products are plain Zarr v3: any Zarr reader works. Here we use `obstore`
(a zagg core dependency) with `skip_signature=True` — unsigned requests against
the public bucket — wrapped in Zarr's `ObjectStore` adapter.

In [ ]:
import numpy as np
import zarr
from obstore.store import S3Store
from zarr.storage import ObjectStore

BUCKET, REGION = "sliderule-public-cors", "us-west-2"


def open_public(prefix):
    s3 = S3Store(BUCKET, prefix=prefix, region=REGION, skip_signature=True)
    return ObjectStore(store=s3, read_only=True)


td_store = open_public("zagg-examples/serc/tdigest.zarr")
gb_store = open_public("zagg-examples/serc/gain_bias.zarr")

# `19` is the child_order group — the ~10 m cell resolution.
td_root = zarr.open_group(td_store, mode="r")["19"]
gb_root = zarr.open_group(gb_store, mode="r")["19"]
print("tdigest arrays:  ", sorted(td_root.array_keys()))
print("gain_bias arrays:", sorted(gb_root.array_keys()))

## 2. The grid and the shardmap (from the repo, no network)

The output grid comes from the run configs shipped **inside zagg**, and the
shardmap + AOI polygon are committed with the repo (`notebooks/data/`,
`tests/data/benchmark/AOP_NEON.geojson`) — the build-once pattern: nothing
here re-queries CMR.

In [ ]:
from importlib.resources import files

from zagg.catalog import load_polygon
from zagg.catalog.shardmap import ShardMap
from zagg.config import load_config
from zagg.grids import from_config

cfg = load_config(str(files("zagg.configs") / "atl03_tdigest_serc.yaml"))
grid = from_config(cfg)
shardmap = ShardMap.from_json("data/sm_serc_healpix_o10.json")
parts = load_polygon("../tests/data/benchmark/AOP_NEON.geojson")

CELLS_PER_SHARD = 4 ** (grid.child_order - grid.parent_order)
print(f"{len(shardmap.shard_keys)} shards x {CELLS_PER_SHARD:,} cells, "
      f"strict-AOI payload: {shardmap.aoi_mask is not None}")

## 3. Photon density and the strict-AOI mask

Each shard is one contiguous slice of the cell index space, so a shard read is
one ranged request per array. Left: every populated ~10 m cell (ICESat-2's
ground tracks). Right: the same cells filtered by the stored `aoi_mask`
boolean — the strict SERC polygon, applied at read time (issue #101's
"package, don't clip").

In [ ]:
import matplotlib.pyplot as plt
from mortie.tools import mort2geo


def shard_slice(root, shard_key, *names):
    lo = grid.block_index(shard_key)[0] * CELLS_PER_SHARD
    return [root[n][lo:lo + CELLS_PER_SHARD] for n in names]


pts = {"lon": [], "lat": [], "count": [], "in_aoi": []}
for key in shardmap.shard_keys:
    count, aoi, mort = shard_slice(td_root, key, "count", "aoi_mask", "morton")
    nz = count > 0
    lat, lon = mort2geo(mort[nz])
    pts["lon"].append(np.where(lon > 180, lon - 360, lon))
    pts["lat"].append(lat)
    pts["count"].append(count[nz])
    pts["in_aoi"].append(aoi[nz])
pts = {k: np.concatenate(v) for k, v in pts.items()}

fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharex=True, sharey=True)
for ax, sel, title in [
    (axes[0], slice(None), f"all populated cells ({len(pts['count']):,})"),
    (axes[1], pts["in_aoi"], f"strict-AOI cells only ({int(pts['in_aoi'].sum()):,})"),
]:
    sc = ax.scatter(pts["lon"][sel], pts["lat"][sel], c=np.log10(pts["count"][sel]),
                    s=1.5, cmap="viridis")
    for lats, lons in parts:
        ax.plot(np.asarray(lons), np.asarray(lats), "r-", lw=1.5)
    ax.set_title(title)
    ax.set_xlabel("longitude")
axes[0].set_ylabel("latitude")
plt.colorbar(sc, ax=axes, label="log10 photons per 10 m cell", shrink=0.8)
plt.show()

## 4. Quantiles from the sketches: terrain and canopy

A cell's t-digest (~256 `(mean, weight)` centroids) evaluates *any* quantile at
read time. Over the SERC forest, the median tracks the surface while the
p95−p05 spread separates closed canopy from open ground — computed here for
the in-AOI cells (>= 20 photons) of the shard with the largest AOI cover,
straight from the published sketches.

In [ ]:
from zagg.csr import iter_csr_cells, read_csr
from zagg.stats.tdigest import quantile_from_tdigest

# The shard most inside the AOI box (largest strict-AOI cell cover).
in_aoi = {
    key: int(grid.aoi_mask_for_children(np.asarray(p, dtype=np.uint64),
                                        grid.children(key)).sum())
    for key, p in zip(shardmap.shard_keys, shardmap.aoi_mask)
}
DENSE_SHARD = max(in_aoi, key=in_aoi.get)
h10 = grid.block_index(DENSE_SHARD)[0]
count, aoi, mort = shard_slice(td_root, DENSE_SHARD, "count", "aoi_mask", "morton")

MIN_PHOTONS = 20     # quantiles are only meaningful with enough samples;
                     # sparser cells are mostly background-noise columns
med, spread, gidx = [], [], []
for h13 in range(h10 * 64, (h10 + 1) * 64):
    for cid, dig in iter_csr_cells(read_csr(td_store, f"19/h_tdigest/{h13}")):
        d = np.asarray(dig, np.float64)
        if d[:, 1].sum() < MIN_PHOTONS:
            continue
        q05, q50, q95 = (quantile_from_tdigest(d, q) for q in (0.05, 0.50, 0.95))
        med.append(q50)
        spread.append(q95 - q05)
        gidx.append((h13 - h10 * 64) * 4096 + cid)
gidx = np.asarray(gidx)
keep = aoi[gidx]
lat, lon = mort2geo(mort[gidx][keep])
lon = np.where(lon > 180, lon - 360, lon)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharex=True, sharey=True)
for ax, vals, label, cmap in [
    (axes[0], np.asarray(med)[keep], "median photon height (m)", "terrain"),
    (axes[1], np.asarray(spread)[keep], "p95 - p05 spread (m)", "magma"),
]:
    # clamp color range to the 2-98th percentile: a few noise-dominated cells
    # otherwise stretch the scale by hundreds of metres
    vlo, vhi = np.percentile(vals, [2, 98])
    sc = ax.scatter(lon, lat, c=vals, s=4, cmap=cmap, vmin=vlo, vmax=vhi)
    plt.colorbar(sc, ax=ax, label=label, shrink=0.85)
    ax.set_xlabel("longitude")
axes[0].set_ylabel("latitude")
fig.suptitle(f"shard {DENSE_SHARD}: quantiles evaluated from the published digests "
             f"({int(keep.sum()):,} in-AOI cells)")
plt.tight_layout()
plt.show()

## 5. Cross-check: t-digest reconstruction vs the stored waveform

The two products were aggregated from identical photons, so rasterizing a
cell's digest over the gain/bias run's own window (`offset_h - 27*gain_h`,
128 × `gain_h`-metre bins) should reproduce the stored `waveform_counts` for
that cell.

In [ ]:
from zagg.readers import rasterize_cell

gain_arr = gb_root["gain_h"][h10 * 64:(h10 + 1) * 64]
off_arr = gb_root["offset_h"][h10 * 64:(h10 + 1) * 64]

# The showcase cell: the most photon-heavy in-AOI cell with a TIGHT return
# (digest q90-q10 < 5 m — surface-dominated, not a noise column), from the
# shard's finest-gain chunks. Fallback: densest in-AOI cell.
populated = [k for k in range(64) if gain_arr[k] > 0]
g_min = min(gain_arr[k] for k in populated)
best = fallback = None
for k in populated:
    if gain_arr[k] > g_min:
        continue
    h13 = h10 * 64 + k
    for cid, dig in iter_csr_cells(read_csr(td_store, f"19/h_tdigest/{h13}")):
        if not aoi[k * 4096 + cid]:
            continue
        d = np.asarray(dig, np.float64)
        w = d[:, 1].sum()
        cand = (w, h13, k, cid, d)
        if fallback is None or w > fallback[0]:
            fallback = cand
        tight = quantile_from_tdigest(d, 0.9) - quantile_from_tdigest(d, 0.1) < 5.0
        if tight and (best is None or w > best[0]):
            best = cand
n_ph, h13, k, cid, dig = best or fallback
gain, off = float(gain_arr[k]), float(off_arr[k])

waveform = gb_root["waveform_counts"][h13 * 4096 + cid]
z_lo = off - 27.0 * gain
recon = rasterize_cell(dig, z_lo, gain, 128)
z = z_lo + gain * (np.arange(128) + 0.5)
r = np.corrcoef(waveform, recon)[0, 1]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.step(z, waveform, where="mid", lw=1.8, label="gain_bias: stored 128-bin waveform")
ax.step(z, recon, where="mid", lw=1.8, alpha=0.75,
        label="t-digest: reconstructed from ~256 centroids")
ax.axvline(off, color="gray", ls=":", lw=1, label="offset_h (DEM anchor, bin 28)")
populated_bins = np.nonzero(waveform + recon > 0)[0]
ax.set_xlim(z[max(populated_bins.min() - 8, 0)], z[min(populated_bins.max() + 8, 127)])
ax.set_xlabel("elevation (m)")
ax.set_ylabel("photons per bin")
ax.set_title(f"chunk {h13}, cell {cid}: {int(n_ph)} photons, {gain} m bins — r = {r:.5f}")
ax.legend()
plt.tight_layout()
plt.show()
print(f"in-window photon totals — waveform: {int(waveform.sum())}, "
      f"digest: {recon.sum():.1f} (cell total {int(n_ph)})")

## 6. Sub-cell structure from the location channel

The located digest stores one packed order-29 morton word (~2 cm cell) per
centroid — where inside the ~10 m cell that centroid's photons sit. Reading it
back is one small CSR array; decoding is pure mortie.

In [ ]:
from zagg.readers import read_locations

locs = next(
    l for m, c, l in read_locations(td_store, "19/h_tdigest")
    if m == h13 and c == cid
)
lat_c, lon_c = mort2geo(locs)
lon_c = np.where(lon_c > 180, lon_c - 360, lon_c)
dx = (lon_c - lon_c.mean()) * 111_320 * np.cos(np.deg2rad(lat_c.mean()))
dy = (lat_c - lat_c.mean()) * 111_320

fig, ax = plt.subplots(figsize=(6.5, 5.5))
sc = ax.scatter(dx, dy, c=dig[:, 0], s=22, cmap="terrain")
plt.colorbar(sc, label="centroid mean height (m)")
ax.set_xlabel("east offset (m)")
ax.set_ylabel("north offset (m)")
ax.set_title(f"cell {cid}: {len(locs)} centroid locations inside one ~10 m cell")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()
print(f"location span: {np.ptp(dx):.1f} m east-west x {np.ptp(dy):.1f} m north-south")

## 7. The published chunk-manifest cache

The first aggregation also left a **granule-keyed chunk-manifest cache** under
`zagg-index/ATL03/007/` (issue #160): per granule, one parquet of HDF5 chunk
byte offsets/sizes + decode metadata. It is grid- and AOI-independent — any
consumer (zagg's `sidecar` index backend via `h5coro-hidefix`, or plain
pandas) can address the archive's raw bytes without walking HDF5 B-trees.

In [ ]:
from io import BytesIO

import obstore
import pandas as pd

idx_store = S3Store(BUCKET, prefix="zagg-index/ATL03/007", region=REGION,
                    skip_signature=True)
listing = obstore.list(idx_store).collect()
print(f"{len(listing)} granule manifests published; first three:")
for meta in listing[:3]:
    print(f"  {meta['path']}  ({meta['size'] / 1024:.0f} KB)")

# manifests are keyed by the granule id STEM (write-back strips ".h5")
granule_id = shardmap.granules[0][0]["id"].removesuffix(".h5")
buf = obstore.get(idx_store, f"{granule_id}.parquet").bytes()
mf = pd.read_parquet(BytesIO(bytes(buf)))
print(f"\n{granule_id}: {len(mf)} chunk records")
mf.head()

## Summary

Four anonymous reads, one public bucket:

- a **density map** of every ~10 m cell ICESat-2 populated over SERC, with the
  strict-AOI mask applied at read time;
- **arbitrary quantiles** (terrain + canopy spread) evaluated from the stored
  t-digests;
- a **byte-level cross-check** of the sketch against the independently
  aggregated 128-bin waveforms;
- **sub-cell centroid locations** from the lossless morton channel, and the
  reusable **chunk-manifest cache** the pipeline left behind.

To rebuild these products from scratch — or aggregate something new from the
same shardmap — start at [`sponsor_end_to_end.ipynb`](sponsor_end_to_end.ipynb).